In [ ]:
# Train BERTopic model
# Note: This notebook is designed to run in Google Colab.

# Install packages
!pip install bertopic
!pip install umap-learn
!pip install hdbscan
!pip install sentence-transformers
!pip install plotly

# Import packages
import pandas as pd
import numpy as np

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance

# 1. Load data
DATA_PATH = "Topic Trends_WholeData_1990-2023.csv"

data = pd.read_csv(DATA_PATH)

timestamps = data["year"].to_list()
abstracts = data["abstract"].astype(str).to_list()

# 2. Extract embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    abstracts,
    show_progress_bar=True
)

np.save("embeddings_all-MiniLM-L6-v2.npy", embeddings)

# 3. Reduce dimensionality
umap_model = UMAP(
    n_neighbors=60,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# 4. Cluster embeddings
hdbscan_model = HDBSCAN(
    min_cluster_size=80,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# 5. Tokenize topics
vectorizer_model = CountVectorizer(
    stop_words="english",
    min_df=2,
    ngram_range=(1, 2)
)

# 6. Create topic representations
ctfidf_model = ClassTfidfTransformer()

representation_model = MaximalMarginalRelevance(
    diversity=0.4
)

# 7. Set BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    top_n_words=15,
    verbose=True
)

# 8. Train model
topics, probs = topic_model.fit_transform(
    abstracts,
    embeddings
)

In [ ]:
# Reduce outliers

# 1. Reduce outliers
new_topics_emb_063 = topic_model.reduce_outliers(
    abstracts,
    topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.63
)

# 2. Update topic model
topic_model_emb_063 = topic_model

topic_model_emb_063.update_topics(
    abstracts,
    topics=new_topics_emb_063
)

# 3. Export topic-level results
topic_info_emb_063 = topic_model_emb_063.get_topic_info()

topic_info_emb_063.to_excel(
    "topic_level_results_emb_063.xlsx",
    index=False
)

# 4. Export document-level results
document_results_emb_063 = data.copy()

document_results_emb_063["original_topic"] = topics
document_results_emb_063["updated_topic"] = new_topics_emb_063

document_results_emb_063["was_outlier"] = document_results_emb_063["original_topic"] == -1
document_results_emb_063["is_still_outlier"] = document_results_emb_063["updated_topic"] == -1
document_results_emb_063["newly_assigned"] = (
    (document_results_emb_063["original_topic"] == -1) &
    (document_results_emb_063["updated_topic"] != -1)
)

document_results_emb_063.to_csv(
    "document_level_topic_assignments_emb_063.csv",
    index=False
)

# 5. Download results
from google.colab import files

files.download("topic_level_results_emb_063.xlsx")
files.download("document_level_topic_assignments_emb_063.csv")

In [ ]:
# 3. Plot topic trends

!pip install -q kaleido

import plotly.graph_objects as go
from google.colab import files

# 1. Add topic labels
custom_topic_labels = {
    0: "Reading",
    1: "Critical studies of race and identity",
    2: "College access",
    3: "Educational leadership",
    4: "Experimental studies of learning",
    5: "Student motivation",
    6: "Math curriculum and instruction",
    7: "Teacher turnover/teacher effectiveness",
    8: "Immigrant students and families",
    9: "Field of educational research/research methods",
    10: "Middle school education",
    11: "Assessment for learning",
    12: "Math learning and problem solving",
    13: "Educational technology",
    14: "Multilingual learners",
    15: "Writing",
    16: "School finance/school choice",
    17: "Special education and inclusive education",
    18: "Randomized controlled trials and effect sizes",
    19: "Early childhood education",
    20: "Science education",
    21: "Bullying and school violence",
    22: "Parent involvement and home schooling",
    23: "Indigenous education",
    24: "Charter schools",
    25: "School segregation",
    26: "Arts education",
    27: "Educational reform",
    28: "Teacher education and professional learning",
    29: "Academic self-concept",
    30: "LGBTQ students and educators",
    31: "STEM education",
    32: "School discipline & exclusion",
    33: "Rural education",
    34: "Classroom discourse",
    35: "Mentoring",
    36: "Literature and literacy development",
    37: "Racial diversity in higher education",
    38: "Student engagement",
    39: "Teacher efficacy",
    40: "Cooperative learning",
    41: "School dropout",
    42: "Self-regulation",
    43: "Data use in education",
    44: "English learner assessment and reclassification"
}

topic_model_emb_063.set_topic_labels(custom_topic_labels)

# 2. Generate topic trends
timestamps = data.year.to_list()

topics_over_time_emb_063 = topic_model_emb_063.topics_over_time(
    abstracts,
    timestamps,
    topics=new_topics_emb_063,
    nr_bins=10,
    datetime_format="%Y"
)

topics_over_time_emb_063["CustomName"] = (
    topics_over_time_emb_063["Topic"].map(custom_topic_labels)
)

# 3. Create percentage trends
topics_over_time_pct_emb_063 = topics_over_time_emb_063.copy()

topics_over_time_pct_emb_063["Percentage"] = (
    topics_over_time_pct_emb_063["Frequency"] /
    topics_over_time_pct_emb_063.groupby("Timestamp")["Frequency"].transform("sum")
) * 100

topics_over_time_pct_plot_emb_063 = topics_over_time_pct_emb_063.copy()
topics_over_time_pct_plot_emb_063["Frequency"] = topics_over_time_pct_plot_emb_063["Percentage"]

# 4. Create plotting groups
plotting_groups = {
    "Curriculum and Instruction I": [6, 12, 13, 20, 31],
    "Curriculum and Instruction II": [0, 11, 15, 26, 34, 36],
    "Diversity, Equity, and Inclusion I": [1, 23, 25, 30, 37],
    "Diversity, Equity, and Inclusion II": [8, 14, 44],
    "Diversity, Equity, and Inclusion III": [17, 32, 33, 41],
    "Learning and Cognitive Development": [4, 5, 38, 29, 35, 42, 40],
    "Teacher Education and Teaching Quality": [7, 28, 39],
    "Educational Policy": [16, 24, 27, 43],
    "Research Methods": [9, 18],
    "Individual Topics": [2, 3, 10, 19, 21, 22]
}

# 5. Plot function
def plot_topic_group(df, topics, title, y_axis_title, file_name):
    fig = go.Figure()

    for topic_id in topics:
        topic_df = df[df["Topic"] == topic_id].sort_values("Timestamp")
        topic_label = custom_topic_labels[topic_id]

        fig.add_trace(
            go.Scatter(
                x=topic_df["Timestamp"],
                y=topic_df["Frequency"],
                mode="lines+markers+text",
                name=topic_label,
                text=[""] * (len(topic_df) - 1) + [topic_label],
                textposition="middle right",
                showlegend=False
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Time period",
        yaxis_title=y_axis_title,
        width=1100,
        height=500,
        margin=dict(l=70, r=300, t=70, b=70),
        template="plotly_white"
    )

    fig.show()
    fig.write_image(file_name, scale=2)

    return file_name

# 6. Plot frequency figures
frequency_files = []

for group_name, topic_ids in plotting_groups.items():
    safe_name = (
        group_name.lower()
        .replace(" ", "_")
        .replace(",", "")
        .replace("&", "and")
    )

    file_name = f"{safe_name}_frequency.png"

    frequency_files.append(
        plot_topic_group(
            df=topics_over_time_emb_063,
            topics=topic_ids,
            title=f"{group_name}: Frequencies",
            y_axis_title="Frequency",
            file_name=file_name
        )
    )

# 7. Plot percentage figures
percentage_files = []

for group_name, topic_ids in plotting_groups.items():
    safe_name = (
        group_name.lower()
        .replace(" ", "_")
        .replace(",", "")
        .replace("&", "and")
    )

    file_name = f"{safe_name}_percentage.png"

    percentage_files.append(
        plot_topic_group(
            df=topics_over_time_pct_plot_emb_063,
            topics=topic_ids,
            title=f"{group_name}: Percentages",
            y_axis_title="Percentage",
            file_name=file_name
        )
    )

# 8. Download figures
for file_name in frequency_files + percentage_files:
    files.download(file_name)